# Lesson 11B: Self-Organizing Maps Practical — MiniSom, U-Matrix, and Component Planes

## Introduction

Lesson 11A derived the SOM update rule and demonstrated it on synthetic blobs and random RGB colors — clean illustrations, but not the real interpretive workflow. This lesson applies `MiniSom` to a real dataset (the UCI **Wine** dataset: 13 chemical measurements from 178 wines across 3 cultivars) and builds the two visualizations that make a trained SOM actually useful for exploratory analysis:

- The **U-Matrix** (unified distance matrix): for every neuron, the average distance to its grid-neighbors' weight vectors. High values mark boundaries between dissimilar regions — the SOM equivalent of seeing where clusters end.
- **Component planes**: one heatmap per input feature, all sharing the same grid layout, showing how each individual feature varies across the map. Nothing else in this course gives you per-feature interpretability on the same fixed 2D projection.

We'll also do what 11A's conclusion promised to check: does a SOM-derived clustering hold up against a direct K-Means clustering of the same data, and how much of the map's neighbor structure actually reflects true input-space distance?

In this lesson:
1. Load and standardize the real Wine dataset
2. Train MiniSom end to end
3. Build and interpret the U-Matrix
4. Build component planes for individual chemical features
5. Compare a two-stage SOM clustering (cluster the neurons, then assign samples) against direct K-Means
6. Quantify topology preservation with MiniSom's topographic error

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [The Wine Dataset](#dataset)
4. [Training MiniSom](#training)
5. [The U-Matrix](#u-matrix)
6. [Component Planes](#component-planes)
7. [SOM-Derived Clusters vs Direct K-Means](#clustering-comparison)
8. [Quantifying Topology Preservation](#topographic-error)
9. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from minisom import MiniSom

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="dataset"></a>
## The Wine Dataset

178 wines, 13 chemical measurements each (alcohol, malic acid, ash, alkalinity, magnesium, phenols, flavanoids, and more), 3 known cultivars. As with every distance-based method so far, we standardize first — a SOM's Euclidean BMU search is just as scale-sensitive as K-Means or DBSCAN. The cultivar labels are used **only for validation** below, never for training.

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target
feature_names = wine.feature_names

X_scaled = StandardScaler().fit_transform(X)

print(f"Wine dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(wine.target_names)} cultivars")
print(f"Features: {feature_names}")
print(f"Cultivar counts: {np.bincount(y)}")

<a name="training"></a>
## Training MiniSom

A 10×10 grid — enough resolution to see interesting sub-structure with only 178 samples, without so many neurons that most sit empty.

In [ ]:
grid_size = 10
som = MiniSom(grid_size, grid_size, X_scaled.shape[1], sigma=3.0, learning_rate=0.5, random_seed=42)
som.random_weights_init(X_scaled)
som.train(X_scaled, 5000, random_order=True)

print(f"Final quantization error: {som.quantization_error(X_scaled):.4f}")

<a name="u-matrix"></a>
## The U-Matrix

`distance_map()` returns exactly the U-Matrix: for each neuron, its average distance to its immediate grid-neighbors, normalized to [0, 1]. We overlay each wine's true cultivar at its BMU (for validation only — the SOM never saw these labels) to check whether U-Matrix boundaries actually separate the three cultivars.

In [ ]:
u_matrix = som.distance_map()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.pcolor(u_matrix.T, cmap='bone_r')
plt.colorbar(im, ax=ax, label='Average distance to grid-neighbors')

markers = ['o', 's', 'D']
colors = ['crimson', 'steelblue', 'forestgreen']
for i, x in enumerate(X_scaled):
    bmu = som.winner(x)
    ax.plot(bmu[0] + 0.5, bmu[1] + 0.5, markers[y[i]], markerfacecolor='none',
            markeredgecolor=colors[y[i]], markersize=10, markeredgewidth=2)

for cultivar, marker, color in zip(wine.target_names, markers, colors):
    ax.plot([], [], marker, markerfacecolor='none', markeredgecolor=color, markeredgewidth=2, label=cultivar)
ax.legend(loc='upper right')
ax.set_title('U-Matrix with True Cultivars Overlaid (labels used for validation only)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Bright (high-distance) regions in the U-Matrix separate the three cultivars, even though the SOM never saw cultivar labels during training")

<a name="component-planes"></a>
## Component Planes

Each component plane is one feature's weight value across the same grid the U-Matrix was drawn on — letting us ask "which chemical property drives the separation we just saw?" Flavanoids and color intensity are classic discriminators for these three cultivars; we plot them alongside alcohol content for comparison.

In [ ]:
features_to_plot = ['flavanoids', 'color_intensity', 'alcohol']
weights = som.get_weights()  # shape (grid_size, grid_size, n_features)

fig, axes = plt.subplots(1, len(features_to_plot), figsize=(6 * len(features_to_plot), 5))
for ax, feature in zip(axes, features_to_plot):
    idx = feature_names.index(feature)
    im = ax.pcolor(weights[:, :, idx].T, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'Component plane: {feature}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Compare these planes to the U-Matrix boundaries above — regions where a component plane changes sharply should line up with U-Matrix boundaries")

<a name="clustering-comparison"></a>
## SOM-Derived Clusters vs Direct K-Means

A SOM alone doesn't output a hard partition — it's common practice to cluster the trained *neurons* (not the raw data) with a second-stage algorithm, then assign each sample the cluster of its BMU. We compare this two-stage approach against running K-Means directly on the standardized data, using Adjusted Rand Index (ARI) against the true cultivar labels as the (validation-only) yardstick.

In [ ]:
neuron_weights = weights.reshape(-1, X_scaled.shape[1])  # (grid_size*grid_size, n_features)
neuron_clusters = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(neuron_weights)
neuron_cluster_grid = neuron_clusters.reshape(grid_size, grid_size)

bmu_indices = np.array([som.winner(x) for x in X_scaled])
som_labels = neuron_cluster_grid[bmu_indices[:, 0], bmu_indices[:, 1]]

kmeans_direct = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_scaled)

ari_som = adjusted_rand_score(y, som_labels)
ari_kmeans = adjusted_rand_score(y, kmeans_direct.labels_)

print(f"Two-stage SOM clustering ARI vs true cultivars: {ari_som:.4f}")
print(f"Direct K-Means clustering ARI vs true cultivars: {ari_kmeans:.4f}")
print("\n💡 Direct K-Means optimizes purely for cluster separation; the SOM route pays a small ARI cost for the topology-preserving, visually-interpretable map it also produces")

<a name="topographic-error"></a>
## Quantifying Topology Preservation

**Topographic error** measures the fraction of samples whose *first* and *second* best-matching units are not adjacent on the grid — a direct, quantitative check of what 11A only showed visually. A low topographic error means the map is faithfully preserving neighborhood structure: points that are close in the original 13-dimensional space are landing on adjacent (or very close) grid cells, not scattered across the map.

In [ ]:
topo_error = som.topographic_error(X_scaled)
print(f"Topographic error: {topo_error:.4f} ({topo_error * len(X_scaled):.0f} of {len(X_scaled)} samples)")
print("💡 A low topographic error confirms the grid is doing its job: nearby inputs really do land on nearby neurons, not just on the same overall map")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **U-Matrix visualizes cluster boundaries directly on the grid** — high-distance "walls" between neurons separate genuinely distinct regions, confirmed here by an unsupervised map correctly isolating three (never-labeled) wine cultivars.
2. **Component planes give per-feature interpretability that a single 2D embedding can't** — you can point to exactly which chemical property drives a given region of the map.
3. **A SOM needs a second clustering step to produce hard cluster labels** — cluster the trained neurons (not the raw data), then assign each sample its BMU's neuron-cluster.
4. **That two-stage route costs a little clustering accuracy relative to direct K-Means** — expected, since the SOM is also solving a harder problem (topology preservation) that K-Means doesn't attempt at all.
5. **Topographic error quantifies what the U-Matrix and component planes only show qualitatively** — how faithfully input-space neighborhoods survive the projection onto the grid.

### Practical Guidance

- Always standardize before training a SOM — the BMU search is Euclidean-distance-based, exactly like K-Means and DBSCAN.
- Use the U-Matrix first to see how many "regions" the map naturally has, before deciding how many clusters to extract from the neurons.
- Read component planes side by side with the U-Matrix — a plane that changes sharply exactly where the U-Matrix shows a boundary is strong evidence that feature is driving the separation.
- Report topographic error (and quantization error) alongside any SOM visualization — a map that looks clean but has high topographic error is misleading, not insightful.

### Next Steps

This closes Lesson 11. In Lesson 12, we move to deep unsupervised learning: autoencoders and variational autoencoders for learned, nonlinear compression.

You now have the complete SOM toolkit: the competitive-learning derivation and topology-preservation proof from 11A, and the production interpretive workflow — U-Matrix, component planes, and topology quantification — from this lesson 🎯